In [69]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
import shap

In [70]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Загрузка данных

In [71]:
drive.mount("/content/drive")
!unzip /content/drive/MyDrive/credit_scoring_data/home-credit-default-risk.zip -d /content/data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/MyDrive/credit_scoring_data/home-credit-default-risk.zip
replace /content/data/HomeCredit_columns_description.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/data/POS_CASH_balance.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace /content/data/application_test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: /content/data/application_test.csv  
replace /content/data/application_train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: /content/data/application_train.csv  
replace /content/data/bureau.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: /content/data/bureau.csv  
replace /content/data/bureau_balance.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: /content/data/bureau_balance.csv  
replace /content/data/credit_card_balance.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:

In [72]:
app = pd.read_csv('/content/data/application_train.csv')
bureau = pd.read_csv('/content/data/bureau.csv')
bur_balance = pd.read_csv('/content/data/bureau_balance.csv')
credit_card_balance = pd.read_csv('/content/data/credit_card_balance.csv')
installments_payments = pd.read_csv('/content/data/installments_payments.csv')
prev_app = pd.read_csv('/content/data/previous_application.csv')
pos_cash_balance = pd.read_csv('/content/data/POS_CASH_balance.csv')



In [73]:
app.shape

(307511, 122)

In [74]:
installments_payments.head(20)

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
5,1137312,164489,1.0,12,-1384.0,-1417.0,5970.375,5970.375
6,2234264,184693,4.0,11,-349.0,-352.0,29432.295,29432.295
7,1818599,111420,2.0,4,-968.0,-994.0,17862.165,17862.165
8,2723183,112102,0.0,14,-197.0,-197.0,70.740,70.740
9,1413990,109741,1.0,4,-570.0,-609.0,14308.470,14308.470


In [75]:
credit_card_balance.head(20)

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.500,0.0,877.500,1700.325,1800.000,1800.000,0.000,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.000,0.0,0.000,2250.000,2250.000,2250.000,60175.080,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.000,0.0,0.000,2250.000,2250.000,2250.000,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.000,0.0,0.000,11795.760,11925.000,11925.000,224949.285,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.000,0.0,11547.000,22924.890,27000.000,27000.000,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0
5,2646502,380010,-7,82903.815,270000,0.0,0.000,0.0,0.000,4449.105,3825.000,3825.000,80519.040,82773.315,82773.315,0.0,0,0.0,0.0,2.0,Active,7,0
6,1079071,171320,-6,353451.645,585000,67500.0,67500.000,0.0,0.000,14684.175,15750.000,15750.000,345433.860,351881.145,351881.145,1.0,1,0.0,0.0,6.0,Active,0,0
7,2095912,118650,-7,47962.125,45000,45000.0,45000.000,0.0,0.000,0.000,264.690,0.000,44735.310,47962.125,47962.125,1.0,1,0.0,0.0,51.0,Active,0,0
8,2181852,367360,-4,291543.075,292500,90000.0,289339.425,0.0,199339.425,130.500,4093.515,4093.515,285376.410,286831.575,286831.575,3.0,8,0.0,5.0,3.0,Active,0,0
9,1235299,203885,-5,201261.195,225000,76500.0,111026.700,0.0,34526.700,6338.340,45000.000,45000.000,192793.275,197224.695,197224.695,3.0,9,0.0,6.0,38.0,Active,0,0


In [76]:
bur_balance.head(10)

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
5,5715448,-5,C
6,5715448,-6,C
7,5715448,-7,C
8,5715448,-8,C
9,5715448,-9,0


# Формирование витрины данных

In [77]:
application = app.copy()
if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(application.columns):
    application["CREDIT_TO_INCOME"] = (
        application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(application.columns):
    application["ANNUITY_TO_INCOME"] = (
        application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

In [79]:
bureau_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        BUREAU_CNT=("SK_ID_BUREAU", "count"),
        BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
        BUREAU_DAYS_CREDIT_AVG=("DAYS_CREDIT", "mean"),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
        BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
        BUREAU_CNT_CREDIT_PROLONG_SUM=("CNT_CREDIT_PROLONG", "sum"),
        BUREAU_AMT_CREDIT_SUM_DEBT_MAX=("AMT_CREDIT_SUM_DEBT", "max"),
        BUREAU_AMT_CREDIT_SUM_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
        BUREAU_AMT_CREDIT_SUM_SUM=("AMT_CREDIT_SUM", "sum"),
    )
    .reset_index()
)

status_cnt = pd.crosstab(bureau["SK_ID_CURR"], bureau["CREDIT_ACTIVE"]).reset_index()
status_cnt = status_cnt.rename(
    columns={
        "Active": "BUREAU_ACTIVE_CNT",
        "Closed": "BUREAU_CLOSED_CNT",
        "Sold": "BUREAU_SOLD_CNT",
        "Bad debt": "BUREAU_BAD_DEBT_CNT",
    }
)

for col in ["BUREAU_ACTIVE_CNT", "BUREAU_CLOSED_CNT", "BUREAU_SOLD_CNT", "BUREAU_BAD_DEBT_CNT"]:
    if col not in status_cnt.columns:
        status_cnt[col] = 0

bureau_agg = bureau_agg.merge(status_cnt, on="SK_ID_CURR", how="left")


In [80]:
prev = prev_app.copy()

if {"AMT_CREDIT", "AMT_APPLICATION"}.issubset(prev.columns):
    prev["CREDIT_APP_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)

if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(prev.columns):
    prev["ANNUITY_CREDIT_RATIO"] = prev["AMT_ANNUITY"] / prev["AMT_CREDIT"].replace(0, np.nan)

prev_app_agg = (
    prev.groupby("SK_ID_CURR")
    .agg(
        PREV_APP_CNT=("SK_ID_PREV", "count"),
        PREV_APP_AMT_CREDIT_SUM=("AMT_CREDIT", "sum"),
        PREV_APP_AMT_CREDIT_MAX=("AMT_CREDIT", "max"),
        PREV_APP_CREDIT_APP_RATIO_AVG=("CREDIT_APP_RATIO", "mean"),
        PREV_APP_ANNUITY_CREDIT_RATIO_AVG=("ANNUITY_CREDIT_RATIO", "mean"),
        PREV_APP_CNT_PAYMENT_AVG=("CNT_PAYMENT", "mean"),
        PREV_APP_LAST_DAYS_DECISION=("DAYS_DECISION", "max"),
        PREV_APP_ANNUITY_CREDIT_AVG=("AMT_ANNUITY", "mean"),
    )
    .reset_index()
)

st = pd.crosstab(prev["SK_ID_CURR"], prev["NAME_CONTRACT_STATUS"])
st = st.rename(
    columns={
        "Approved": "PREV_APP_APPROVED_CNT",
        "Refused": "PREV_APP_REFUSED_CNT",
        "Canceled": "PREV_APP_CANCELED_CNT",
        "Unused offer": "PREV_APP_UNUSED_OFFER_CNT",
    }
)

for col in [
    "PREV_APP_APPROVED_CNT",
    "PREV_APP_REFUSED_CNT",
    "PREV_APP_CANCELED_CNT",
    "PREV_APP_UNUSED_OFFER_CNT",
]:
    if col not in st.columns:
        st[col] = 0

st["PREV_APP_TOTAL_CNT"] = st[
    ["PREV_APP_APPROVED_CNT", "PREV_APP_REFUSED_CNT", "PREV_APP_CANCELED_CNT", "PREV_APP_UNUSED_OFFER_CNT"]
].sum(axis=1)

denom = st["PREV_APP_TOTAL_CNT"].replace(0, np.nan)
st["PREV_APP_APPROVED_RATE"] = st["PREV_APP_APPROVED_CNT"] / denom
st["PREV_APP_REFUSED_RATE"] = st["PREV_APP_REFUSED_CNT"] / denom

st = st.reset_index()[["SK_ID_CURR", "PREV_APP_APPROVED_RATE", "PREV_APP_REFUSED_RATE"]]

In [81]:
installments_payments["DPD"] = installments_payments["DAYS_ENTRY_PAYMENT"] - installments_payments["DAYS_INSTALMENT"]

installments_payments["DBD"] = installments_payments["DAYS_INSTALMENT"] - installments_payments["DAYS_ENTRY_PAYMENT"]

# индикаторы просрочек
installments_payments["DPD_POS"] = (installments_payments["DPD"] > 0).astype(int)
installments_payments["DPD_30"] = (installments_payments["DPD"] > 30).astype(int)
installments_payments["DPD_90"] = (installments_payments["DPD"] > 90).astype(int)

# недоплата
installments_payments["PAYMENT_DIFF"] = installments_payments["AMT_PAYMENT"] - installments_payments["AMT_INSTALMENT"]
installments_payments["UNDERPAY"] = (installments_payments["PAYMENT_DIFF"] < 0).astype(int)

installments_agg = (
    installments_payments.groupby("SK_ID_CURR")
    .agg(
        INST_CNT=("SK_ID_PREV", "count"),

        INST_DPD_MAX=("DPD", "max"),
        INST_DPD_MEAN=("DPD", "mean"),
        INST_DPD_SUM=("DPD", "sum"),

        INST_DPD_POS_CNT=("DPD_POS", "sum"),
        INST_DPD_30_CNT=("DPD_30", "sum"),
        INST_DPD_90_CNT=("DPD_90", "sum"),

        INST_DPD_POS_RATIO=("DPD_POS", "mean"),
        INST_DPD_30_RATIO=("DPD_30", "mean"),
        INST_DPD_90_RATIO=("DPD_90", "mean"),

        INST_PAYMENT_DIFF_MEAN=("PAYMENT_DIFF", "mean"),
        INST_PAYMENT_DIFF_MIN=("PAYMENT_DIFF", "min"),

        INST_UNDERPAY_CNT=("UNDERPAY", "sum"),
        INST_UNDERPAY_RATIO=("UNDERPAY", "mean"),

        INST_AMT_INSTALMENT_MEAN=("AMT_INSTALMENT", "mean"),
        INST_AMT_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
    )
    .reset_index()
)

In [82]:
# Утилизация лимита
credit_card_balance["CC_UTIL"] = (
    credit_card_balance["AMT_BALANCE"] /
    credit_card_balance["AMT_CREDIT_LIMIT_ACTUAL"]
)

credit_card_balance["CC_UTIL"] = credit_card_balance["CC_UTIL"].replace([np.inf, -np.inf], np.nan)

# Просрочка > 30 дней
credit_card_balance["CC_DPD_30"] = (
    credit_card_balance["SK_DPD"] > 30
).astype(int)


# Разница с минимальным платежом
credit_card_balance["CC_PAY_GAP"] = (
    credit_card_balance["AMT_PAYMENT_CURRENT"] -
    credit_card_balance["AMT_INST_MIN_REGULARITY"]
)


# Сортировка для расчёта тренда
credit_card_balance = credit_card_balance.sort_values(
    ["SK_ID_CURR", "MONTHS_BALANCE"]
)

# Изменение баланса кредитной карты
credit_card_balance["CC_BALANCE_DELTA"] = (
    credit_card_balance
        .groupby("SK_ID_CURR")["AMT_BALANCE"]
        .diff()
)
credit_card_agg = (
    credit_card_balance.groupby("SK_ID_CURR")
    .agg(
        CC_UTIL_MEAN=("CC_UTIL", "mean"),
        CC_DPD_30_RATIO=("CC_DPD_30", "mean"),
        CC_PAY_GAP_MEAN=("CC_PAY_GAP", "mean"),
        CC_BALANCE_TREND=("CC_BALANCE_DELTA", "mean"),
    )
    .reset_index()
)

In [83]:
df_final = (
    application
    .merge(bureau_agg, on="SK_ID_CURR", how="left")
    .merge(prev_app_agg, on="SK_ID_CURR", how="left")
    .merge(st, on="SK_ID_CURR", how="left")
    .merge(installments_agg, on="SK_ID_CURR", how="left")
    .merge(credit_card_agg, on="SK_ID_CURR", how="left")
)

df_final.shape
df_final, df_final_val = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final["TARGET"])

In [84]:
df_final.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_A

# Предобработка данных

In [85]:
corr_matrix = df_final.corr(numeric_only=True)
corr_pairs = (
    corr_matrix
    .abs()
    .unstack()
    .sort_values(ascending=False)
)

corr_pairs = corr_pairs[corr_pairs < 1]
corr_pairs[corr_pairs > 0.6]

,,0
OBS_60_CNT_SOCIAL_CIRCLE,OBS_30_CNT_SOCIAL_CIRCLE,0.998514
OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.998514
YEARS_BUILD_MEDI,YEARS_BUILD_AVG,0.998391
YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998391
FLOORSMIN_AVG,FLOORSMIN_MEDI,0.997322
FLOORSMIN_MEDI,FLOORSMIN_AVG,0.997322
FLOORSMAX_AVG,FLOORSMAX_MEDI,0.996983
FLOORSMAX_MEDI,FLOORSMAX_AVG,0.996983
ENTRANCES_MEDI,ENTRANCES_AVG,0.996911
ENTRANCES_AVG,ENTRANCES_MEDI,0.996911


In [86]:
cols_to_drop = [

    # MEDI / MODE дубли
    "YEARS_BUILD_MEDI", "YEARS_BUILD_MODE",
    "FLOORSMIN_MEDI", "FLOORSMIN_MODE",
    "FLOORSMAX_MEDI", "FLOORSMAX_MODE",
    "ENTRANCES_MEDI", "ENTRANCES_MODE",
    "ELEVATORS_MEDI", "ELEVATORS_MODE",
    "COMMONAREA_MEDI", "COMMONAREA_MODE",
    "LIVINGAREA_MEDI", "LIVINGAREA_MODE",
    "APARTMENTS_MEDI", "APARTMENTS_MODE",
    "BASEMENTAREA_MEDI", "BASEMENTAREA_MODE",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAPARTMENTS_MODE",
    "LANDAREA_MEDI", "LANDAREA_MODE",
    "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAPARTMENTS_MODE",
    "NONLIVINGAREA_MEDI", "NONLIVINGAREA_MODE",
    "YEARS_BEGINEXPLUATATION_MEDI", "YEARS_BEGINEXPLUATATION_MODE",
    "FLOORSMAX_AVG",

    # Соц. окружение
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",

    # Регион
    "REGION_RATING_CLIENT_W_CITY",

    # Bureau
    "BUREAU_AMT_CREDIT_SUM_DEBT_MAX",
    "BUREAU_CLOSED_CNT",

    # Семья
    "CNT_CHILDREN",

    # Финансовые
    "AMT_GOODS_PRICE",

    # Previous application
    "PREV_APP_AMT_CREDIT_MAX",
    "APARTMENTS_AVG",
    "LIVINGAPARTMENTS_AVG",
    "TOTALAREA_MODE",
    "ELEVATORS_AVG",

    # Региональные дубли
    "LIVE_REGION_NOT_WORK_REGION",
    "LIVE_CITY_NOT_WORK_CITY",

    # Финансовые ratio (оставляем CREDIT_TO_INCOME)
    "ANNUITY_TO_INCOME",

    #Агрегаты по платежам
    "INST_AMT_INSTALMENT_MEAN",
    "INST_DPD_30_CNT",
    "INST_DPD_POS_RATIO",
    "INST_UNDERPAY_CNT"
]

df_final = df_final.drop(columns=cols_to_drop, errors="ignore")

In [87]:
corr_matrix = df_final.corr(numeric_only=True)
corr_pairs = (
    corr_matrix
    .abs()
    .unstack()
    .sort_values(ascending=False)
)

corr_pairs = corr_pairs[corr_pairs < 1]
corr_pairs[corr_pairs > 0.7]

,,0
AMT_CREDIT,AMT_ANNUITY,0.770163
AMT_ANNUITY,AMT_CREDIT,0.770163


In [88]:
df_final.isna().mean().sort_values(ascending=False)

,0
CC_PAY_GAP_MEAN,0.801027
CC_UTIL_MEAN,0.719607
CC_BALANCE_TREND,0.718782
CC_DPD_30_RATIO,0.716745
COMMONAREA_AVG,0.698396
NONLIVINGAPARTMENTS_AVG,0.693998
FONDKAPREMONT_MODE,0.683779
FLOORSMIN_AVG,0.678519
YEARS_BUILD_AVG,0.664787
OWN_CAR_AGE,0.660214


In [89]:
cols_keep = [
    "CC_PAY_GAP_MEAN",
    "CC_UTIL_MEAN",
    "CC_BALANCE_TREND",
    "CC_DPD_30_RATIO",
]

missing_ratio = df_final.isna().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.60].index.tolist()

cols_to_drop = [c for c in cols_to_drop if c not in cols_keep]

df_final = df_final.drop(columns=cols_to_drop)

print("Удалено признаков:", len(cols_to_drop))
print(cols_to_drop)

Удалено признаков: 6
['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE']


In [90]:
df_final = df_final.drop(columns=["CODE_GENDER"], errors="ignore") # удаляем, так как это дискриминация по половому признаку

In [91]:
df_final.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'ENTRANCES_AVG',
 'LANDAREA_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAREA_AVG',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE',
 'OBS_30_CNT_SOCIAL_CIRCLE',
 'DE

In [92]:
df_final["TARGET"].mean() # доля дефолтов в датасете

np.float64(0.08072908198107379)

In [93]:
def is_binary_numeric(series):                   # проверка, какие признаки бинарные
    unique_vals = set(series.dropna().unique())
    return unique_vals.issubset({0, 1})
for col in df_final.columns:
  if is_binary_numeric(df_final[col]):
    print(col)

TARGET
FLAG_MOBIL
FLAG_EMP_PHONE
FLAG_WORK_PHONE
FLAG_CONT_MOBILE
FLAG_PHONE
FLAG_EMAIL
REG_REGION_NOT_LIVE_REGION
REG_REGION_NOT_WORK_REGION
REG_CITY_NOT_LIVE_CITY
REG_CITY_NOT_WORK_CITY
FLAG_DOCUMENT_2
FLAG_DOCUMENT_3
FLAG_DOCUMENT_4
FLAG_DOCUMENT_5
FLAG_DOCUMENT_6
FLAG_DOCUMENT_7
FLAG_DOCUMENT_8
FLAG_DOCUMENT_9
FLAG_DOCUMENT_10
FLAG_DOCUMENT_11
FLAG_DOCUMENT_12
FLAG_DOCUMENT_13
FLAG_DOCUMENT_14
FLAG_DOCUMENT_15
FLAG_DOCUMENT_16
FLAG_DOCUMENT_17
FLAG_DOCUMENT_18
FLAG_DOCUMENT_19
FLAG_DOCUMENT_20
FLAG_DOCUMENT_21
BUREAU_BAD_DEBT_CNT


# Биннинг, WOE encoding и подсчет IV для каждого признака

In [94]:
df_work = df_final.copy()
target = "TARGET"

# drop id
if "SK_ID_CURR" in df_work.columns:
    df_work = df_work.drop(columns=["SK_ID_CURR"])

binary_cols = [c for c in df_work.columns if is_binary_numeric(df_work[c]) and c != target]


numeric_cols = df_work.select_dtypes(include=[np.number]).columns.tolist()
if target in numeric_cols:
    numeric_cols.remove(target)

categorical_cols = df_work.select_dtypes(include=["object"]).columns.tolist()

numeric_cols = [c for c in numeric_cols if c not in binary_cols]
categorical_cols = categorical_cols + binary_cols

categorical_cols = list(dict.fromkeys(categorical_cols))

def quantile_binning(series, q=10):
    s = series.copy()
    mask_na = s.isna()
    try:
        binned = pd.qcut(s[~mask_na], q=q, duplicates="drop")
    except Exception:
        binned = pd.cut(s[~mask_na], bins=q)
    out = pd.Series(index=s.index, dtype="object")
    out.loc[~mask_na] = binned.astype(str)
    out.loc[mask_na] = "__MISSING__"
    return out

binned_df = df_work.copy()

for col in numeric_cols:
    binned_df[col] = quantile_binning(df_work[col], q=10)

# категориальные + бинарные оставляем, но NaN пометим
for col in categorical_cols:
    binned_df[col] = binned_df[col].astype("object").where(binned_df[col].notna(), "__MISSING__")

# WOE/IV
def compute_woe_iv(df, feature, target):
    eps = 1e-6
    g = df.groupby(feature, dropna=False)[target].agg(["count", "sum"])
    g.columns = ["total", "bad"]
    g["good"] = g["total"] - g["bad"]

    total_good = g["good"].sum()
    total_bad = g["bad"].sum()

    g["dist_good"] = g["good"] / (total_good + eps)
    g["dist_bad"] = g["bad"] / (total_bad + eps)

    g["WOE"] = np.log((g["dist_good"] + eps) / (g["dist_bad"] + eps))
    g["IV"] = (g["dist_good"] - g["dist_bad"]) * g["WOE"]
    return g["WOE"].to_dict(), float(g["IV"].sum())

woe_df = pd.DataFrame(index=df_work.index)
iv_table = []

all_features = numeric_cols + categorical_cols

for col in all_features:
    woe_map, iv = compute_woe_iv(binned_df[[col, target]], col, target)
    iv_table.append({"feature": col, "IV": iv})
    woe_df[col] = binned_df[col].map(woe_map)

woe_df[target] = df_work[target]
iv_df = pd.DataFrame(iv_table).sort_values("IV", ascending=False)

iv_df.head(20)

/tmp/ipython-input-3730801063.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  woe_df[col] = binned_df[col].map(woe_map)
/tmp/ipython-input-3730801063.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  woe_df[col] = binned_df[col].map(woe_map)
/tmp/ipython-input-3730801063.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, 

,feature,IV
13,EXT_SOURCE_3,0.329764
12,EXT_SOURCE_2,0.303195
11,EXT_SOURCE_1,0.151344
32,BUREAU_DAYS_CREDIT_AVG,0.119970
5,DAYS_EMPLOYED,0.111552
4,DAYS_BIRTH,0.086772
31,BUREAU_DAYS_CREDIT_MAX,0.083580
73,OCCUPATION_TYPE,0.083309
75,ORGANIZATION_TYPE,0.075219
42,PREV_APP_CREDIT_APP_RATIO_AVG,0.070548


In [95]:
# df_final = df_final.drop(columns=["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], errors="ignore") # для честного скоринга не будем учитывать внешние скоринги, так как они отчасти дают "утечку" в данных
# woe_df = woe_df.drop(columns=["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], errors="ignore")
non_informative_features = iv_df[iv_df["IV"] < 0.015]                                               # удаляем неинформативные по IV колонки

df_final = df_final.drop(columns=non_informative_features["feature"].tolist(), errors="ignore")
woe_df = woe_df.drop(columns=non_informative_features["feature"].tolist(), errors="ignore")
print(f"Удалено неинформативных признаков {len(non_informative_features)}: {non_informative_features}")

Удалено неинформативных признаков 56:                                feature        IV
26           AMT_REQ_CREDIT_BUREAU_MON  0.014877
65                  NAME_CONTRACT_TYPE  0.014871
30                          BUREAU_CNT  0.014528
45         PREV_APP_LAST_DAYS_DECISION  0.014302
27           AMT_REQ_CREDIT_BUREAU_QRT  0.013533
25          AMT_REQ_CREDIT_BUREAU_WEEK  0.013245
24           AMT_REQ_CREDIT_BUREAU_DAY  0.013245
23          AMT_REQ_CREDIT_BUREAU_HOUR  0.013245
41             PREV_APP_AMT_CREDIT_SUM  0.013005
93                     FLAG_DOCUMENT_6  0.012110
109                BUREAU_BAD_DEBT_CNT  0.011900
34   BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM  0.011690
35       BUREAU_CNT_CREDIT_PROLONG_SUM  0.011690
39                     BUREAU_SOLD_CNT  0.011690
33       BUREAU_CREDIT_DAY_OVERDUE_MAX  0.011690
0                     AMT_INCOME_TOTAL  0.011271
49                            INST_CNT  0.011255
29                    CREDIT_TO_INCOME  0.011170
10             HOUR_APPR_PROCES

In [96]:
woe_df.shape

(246008, 55)

# Считаем VIF для выявления мультиколлинеарности

In [97]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = woe_df.drop(columns=["TARGET"])
X = sm.add_constant(X)
vif_df = pd.DataFrame()
vif_df["features"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]
vif_df.sort_values("VIF", ascending=False).head(20)

,features,VIF
48,HOUSETYPE_MODE,8.724090
50,EMERGENCYSTATE_MODE,7.567569
14,ENTRANCES_AVG,6.741489
16,LIVINGAREA_AVG,5.489713
49,WALLSMATERIAL_MODE,4.767690
37,INST_UNDERPAY_RATIO,4.764548
13,YEARS_BEGINEXPLUATATION_AVG,4.689055
31,INST_DPD_MAX,4.654018
36,INST_PAYMENT_DIFF_MIN,4.327553
17,NONLIVINGAREA_AVG,4.317090


In [98]:
high_vif = vif_df[vif_df["VIF"] > 5]
df_final = df_final.drop(columns=high_vif["features"], errors='ignore')
woe_df = woe_df.drop(columns=high_vif["features"], errors='ignore')
print(f"Удалено мультиколлинеарных признаков {len(high_vif)}: {high_vif}")
woe_df.shape

Удалено мультиколлинеарных признаков 4:                features       VIF
14        ENTRANCES_AVG  6.741489
16       LIVINGAREA_AVG  5.489713
48       HOUSETYPE_MODE  8.724090
50  EMERGENCYSTATE_MODE  7.567569


(246008, 51)

In [99]:
woe_df.columns.tolist()

['AMT_CREDIT',
 'AMT_ANNUITY',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'REGION_RATING_CLIENT',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'LANDAREA_AVG',
 'NONLIVINGAREA_AVG',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'BUREAU_DAYS_CREDIT_MAX',
 'BUREAU_DAYS_CREDIT_AVG',
 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_AMT_CREDIT_SUM_SUM',
 'BUREAU_ACTIVE_CNT',
 'PREV_APP_CREDIT_APP_RATIO_AVG',
 'PREV_APP_ANNUITY_CREDIT_RATIO_AVG',
 'PREV_APP_CNT_PAYMENT_AVG',
 'PREV_APP_ANNUITY_CREDIT_AVG',
 'PREV_APP_APPROVED_RATE',
 'PREV_APP_REFUSED_RATE',
 'INST_DPD_MAX',
 'INST_DPD_MEAN',
 'INST_DPD_SUM',
 'INST_DPD_POS_CNT',
 'INST_PAYMENT_DIFF_MEAN',
 'INST_PAYMENT_DIFF_MIN',
 'INST_UNDERPAY_RATIO',
 'INST_AMT_PAYMENT_MEAN',
 'CC_UTIL_MEAN',
 'CC_PAY_GAP_MEAN',
 'CC_BALANCE_TREND',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_H

# Оцениваем ранжирующую способность признаков по коэффициенту Джини


In [100]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
def univariate_gini_table(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    rows = []

    for col in X_train.columns:
        s_tr = X_train[col].values
        s_te = X_test[col].values

        # если константа — пропускаем
        if np.std(s_tr) < 1e-12:
            continue

        auc_tr = roc_auc_score(y_train, s_tr)
        auc_te = roc_auc_score(y_test, s_te)

        rows.append({
            "feature": col,
            "gini_train": abs(2 * auc_tr - 1),
            "gini_test": abs(2 * auc_te - 1),
            "gini_drop": abs(2 * auc_tr - 1) - abs(2 * auc_te - 1)
        })

    return pd.DataFrame(rows).sort_values(
        "gini_train", ascending=False
    ).reset_index(drop=True)
gini_df = univariate_gini_table(woe_df.drop(columns=["TARGET"]), woe_df["TARGET"])
gini_df.head(20)

,feature,gini_train,gini_test,gini_drop
0,EXT_SOURCE_3,0.316787,0.305219,0.011568
1,EXT_SOURCE_2,0.303375,0.306685,-0.003311
2,BUREAU_DAYS_CREDIT_AVG,0.197490,0.183996,0.013494
3,DAYS_EMPLOYED,0.187226,0.183818,0.003408
4,EXT_SOURCE_1,0.177855,0.167800,0.010056
5,DAYS_BIRTH,0.169367,0.155598,0.013768
6,BUREAU_DAYS_CREDIT_MAX,0.163765,0.158832,0.004933
7,OCCUPATION_TYPE,0.152961,0.151887,0.001073
8,ORGANIZATION_TYPE,0.151233,0.150093,0.001140
9,PREV_APP_CREDIT_APP_RATIO_AVG,0.146750,0.151589,-0.004838


In [101]:
low_gini = gini_df[gini_df['gini_train']<0.05]
df_final = df_final.drop(columns=low_gini["feature"], errors='ignore')
woe_df = woe_df.drop(columns=low_gini["feature"], errors='ignore')
print(f"Удалено признаков c коэффициентом Джини <0.05 {len(low_gini)}: {low_gini["feature"]}")
woe_df.shape

Удалено признаков c коэффициентом Джини <0.05 2: 48    REG_CITY_NOT_LIVE_CITY
49         NAME_HOUSING_TYPE
Name: feature, dtype: object


(246008, 49)

In [102]:
woe_df.columns.tolist()

['AMT_CREDIT',
 'AMT_ANNUITY',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'REGION_RATING_CLIENT',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'LANDAREA_AVG',
 'NONLIVINGAREA_AVG',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'BUREAU_DAYS_CREDIT_MAX',
 'BUREAU_DAYS_CREDIT_AVG',
 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_AMT_CREDIT_SUM_SUM',
 'BUREAU_ACTIVE_CNT',
 'PREV_APP_CREDIT_APP_RATIO_AVG',
 'PREV_APP_ANNUITY_CREDIT_RATIO_AVG',
 'PREV_APP_CNT_PAYMENT_AVG',
 'PREV_APP_ANNUITY_CREDIT_AVG',
 'PREV_APP_APPROVED_RATE',
 'PREV_APP_REFUSED_RATE',
 'INST_DPD_MAX',
 'INST_DPD_MEAN',
 'INST_DPD_SUM',
 'INST_DPD_POS_CNT',
 'INST_PAYMENT_DIFF_MEAN',
 'INST_PAYMENT_DIFF_MIN',
 'INST_UNDERPAY_RATIO',
 'INST_AMT_PAYMENT_MEAN',
 'CC_UTIL_MEAN',
 'CC_PAY_GAP_MEAN',
 'CC_BALANCE_TREND',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'OCCUPA

# Обучение логистической регрессии (для интерпретируемости)

In [103]:
FEATURES = woe_df.columns.tolist()

def compute_woe_map(df_binned: pd.DataFrame, feature: str, target: str, eps: float = 1e-6):
    g = df_binned.groupby(feature, dropna=False)[target].agg(["count", "sum"])
    g.columns = ["total", "bad"]
    g["good"] = g["total"] - g["bad"]

    total_good = g["good"].sum()
    total_bad  = g["bad"].sum()

    g["dist_good"] = g["good"] / (total_good + eps)
    g["dist_bad"]  = g["bad"]  / (total_bad  + eps)

    woe = np.log((g["dist_good"] + eps) / (g["dist_bad"] + eps))
    return woe.to_dict()

def _is_binary_numeric(s: pd.Series) -> bool:
    if not pd.api.types.is_numeric_dtype(s):
        return False
    u = pd.Series(s.dropna().unique())
    return len(u) <= 2 and set(u.tolist()).issubset({0, 1})

def fit_woe_encoder(df_train: pd.DataFrame, features=FEATURES, target: str = "TARGET", q: int = 10):
    # 0) оставляем только нужные колонки (и target)
    cols = [c for c in features if c in df_train.columns]
    df = df_train[cols].copy()

    # 1) типы
    feats = [c for c in cols if c != target]
    binary_cols = [c for c in feats if _is_binary_numeric(df[c])]
    numeric_cols = [c for c in feats if pd.api.types.is_numeric_dtype(df[c]) and c not in binary_cols]
    categorical_cols = [c for c in feats if c not in numeric_cols and c not in binary_cols] + binary_cols
    categorical_cols = list(dict.fromkeys(categorical_cols))

    # 2) fit "qcut edges" на train для numeric
    edges = {}
    for col in numeric_cols:
        s = df[col].astype(float)
        x = s.dropna().values
        if len(x) == 0:
            edges[col] = None
            continue
        qs = np.unique(np.nanquantile(x, np.linspace(0, 1, q + 1)))
        # если мало уникальных квантилей — биннинг в qcut развалится, fallback: равные bins по min/max
        if len(qs) < 3:
            mn, mx = np.nanmin(x), np.nanmax(x)
            if np.isfinite(mn) and np.isfinite(mx) and mn < mx:
                qs = np.linspace(mn, mx, q + 1)
            else:
                qs = None
        edges[col] = qs

    # 3) бинним train и считаем woe_map
    binned = pd.DataFrame(index=df.index)
    for col in numeric_cols:
        s = df[col].astype(float)
        mask_na = s.isna()
        out = pd.Series(index=df.index, dtype="object")
        if edges[col] is None:
            out.loc[~mask_na] = "__ALL__"
        else:
            out.loc[~mask_na] = pd.cut(s.loc[~mask_na], bins=edges[col], include_lowest=True).astype(str)
        out.loc[mask_na] = "__MISSING__"
        binned[col] = out

    for col in categorical_cols:
        binned[col] = df[col].astype("object").where(df[col].notna(), "__MISSING__")

    # 4) WOE maps (только по train)
    woe_maps = {}
    for col in (numeric_cols + categorical_cols):
        tmp = pd.DataFrame({col: binned[col], target: df[target].astype(int)})
        woe_maps[col] = compute_woe_map(tmp, col, target)

    meta = {
        "target": target,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "edges": edges,
        "woe_maps": woe_maps,
        "features_order": feats,  # порядок фичей без target
    }
    return meta

def transform_woe(df_any: pd.DataFrame, meta: dict, features=FEATURES, neutral_woe: float = 0.0):
    target = meta["target"]
    cols = [c for c in features if c in df_any.columns]
    df = df_any[cols].copy()

    # бинним по train-границам
    binned = pd.DataFrame(index=df.index)

    for col in meta["numeric_cols"]:
        if col not in df.columns:
            continue
        s = df[col].astype(float)
        mask_na = s.isna()
        out = pd.Series(index=df.index, dtype="object")
        ed = meta["edges"].get(col)
        if ed is None:
            out.loc[~mask_na] = "__ALL__"
        else:
            out.loc[~mask_na] = pd.cut(s.loc[~mask_na], bins=ed, include_lowest=True).astype(str)
        out.loc[mask_na] = "__MISSING__"
        binned[col] = out

    for col in meta["categorical_cols"]:
        if col not in df.columns:
            continue
        binned[col] = df[col].astype("object").where(df[col].notna(), "__MISSING__")

    woe_df = pd.DataFrame(index=df.index)
    for col in meta["features_order"]:
        if col not in binned.columns:
            woe_df[col] = neutral_woe
            continue
        wmap = meta["woe_maps"].get(col, {})
        woe_df[col] = binned[col].map(wmap).fillna(neutral_woe)

    if target in df.columns:
        woe_df[target] = df[target].astype(int).values

    ordered = [c for c in FEATURES if c != "TARGET"]
    woe_df = woe_df.reindex(columns=ordered + ([target] if target in woe_df.columns else []))
    return woe_df
meta = fit_woe_encoder(df_final, q=10)
woe_df_val = transform_woe(df_final_val, meta)


In [104]:
from sklearn.linear_model import LogisticRegression
X_train, y_train = woe_df.drop(columns=["TARGET"]), woe_df["TARGET"]
X_val, y_val = woe_df_val.drop(columns=["TARGET"]), woe_df_val["TARGET"]
model = LogisticRegression(penalty="l2", random_state=42)
model.fit(X_train, y_train)
proba_train = model.predict_proba(X_train)[:, 1]
proba_val = model.predict_proba(X_val)[:, 1]
auc_train = roc_auc_score(y_train, proba_train)
auc_val = roc_auc_score(y_val, proba_val)
print("AUC for train:", auc_train)
print("AUC for validation:", auc_val)

AUC for train: 0.7549988040431811
AUC for validation: 0.7574009256560756


In [105]:
!pip install catboost

In [106]:
from catboost import CatBoostClassifier, Pool
cat_model = CatBoostClassifier(iterations=1000,
                           learning_rate=0.05,
                           max_depth=6,
                           loss_function='Logloss',
                           early_stopping_rounds=200,
                           eval_metric='AUC',
                           verbose=200)
train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)
cat_model.fit(train_pool, eval_set=val_pool, use_best_model=True)


0:	test: 0.6080233	best: 0.6080233 (0)	total: 30.8ms	remaining: 30.8s
200:	test: 0.7653632	best: 0.7653632 (200)	total: 4.78s	remaining: 19s
400:	test: 0.7690690	best: 0.7690886 (399)	total: 9.45s	remaining: 14.1s
600:	test: 0.7703633	best: 0.7704078 (597)	total: 14.9s	remaining: 9.92s
800:	test: 0.7707143	best: 0.7707643 (786)	total: 19.5s	remaining: 4.85s
999:	test: 0.7709012	best: 0.7709805 (970)	total: 24s	remaining: 0us

bestTest = 0.7709804886
bestIteration = 970

Shrink model to first 971 iterations.


CatBoostClassifier(early_stopping_rounds=200, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', max_depth=6, verbose=200)

# Wald-тест для каждого признака

In [107]:
X_sm = sm.add_constant(X_train, has_constant="add")
model = sm.Logit(y_train, X_train)
res = model.fit(disp=False)

wald_table = pd.DataFrame({
    "coef": res.params,
    "se": res.bse,
    "z": res.tvalues,
    "p_value": res.pvalues
}).sort_values("p_value")

wald_table.sort_values("coef", ascending=False)

,coef,se,z,p_value
AMT_REQ_CREDIT_BUREAU_YEAR,0.817802,0.054831,14.914889,2.637138e-50
INST_PAYMENT_DIFF_MIN,0.778747,0.054940,14.174580,1.316288e-45
FLAG_EMP_PHONE,0.696658,0.046709,14.914945,2.634943e-50
INST_DPD_MEAN,0.422006,0.044590,9.464116,2.960544e-21
PREV_APP_ANNUITY_CREDIT_AVG,0.213162,0.041273,5.164732,2.407835e-07
INST_DPD_POS_CNT,0.185072,0.051343,3.604616,3.126145e-04
PREV_APP_ANNUITY_CREDIT_RATIO_AVG,0.166971,0.049230,3.391629,6.947838e-04
LANDAREA_AVG,0.118673,0.066086,1.795724,7.253847e-02
NONLIVINGAREA_AVG,0.061923,0.057588,1.075284,2.822476e-01
DAYS_BIRTH,0.027486,0.022551,1.218871,2.228933e-01
